In [4]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".joblib"):
            print(os.path.join(dirname, filename))

/kaggle/input/models/danilzhukovv/ecup-baseline-logreg-l12/scikitlearn/default/1/baseline_logreg_l12.joblib


In [1]:
print(1)

1


In [2]:
import os, re, gc, json, time
os.environ["TOKENIZERS_PARALLELISM"]="true"
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

from pathlib import Path
import numpy as np, pandas as pd, polars as pl, torch
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from catboost import CatBoostClassifier, Pool

ROOT, OUT = Path("/kaggle/input"), Path("/kaggle/working/conflict_expert")
OUT.mkdir(exist_ok=True)

def find_file(name, required=True):
    p=sorted(ROOT.rglob(name), key=lambda x:("hakaton-ozon-math-items" not in str(x),len(str(x))))
    if p:return p[0]
    if required:raise FileNotFoundError(name)

MATCHES=find_file("matches.parquet")
ITEMS=find_file("items_human.parquet",False) or find_file("items.parquet")
TEST_MATCHES=find_file("matches_test.parquet",False)
TEST_ITEMS=find_file("items_test.parquet",False)

def find_model():
    for p in ROOT.rglob("config.json"):
        try:
            c=json.loads(p.read_text())
            if c.get("vocab_size")==46166 and c.get("hidden_size")==1024 and c.get("num_hidden_layers")==24 and (p.parent/"model.safetensors").exists():
                return p.parent
        except: pass
    raise FileNotFoundError("Не найдена BGE-M3 vocab-trim 46k")
MODEL=find_model()
print("GPU:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("train:",MATCHES,ITEMS,"\nmodel:",MODEL,"\ntest:",TEST_MATCHES,TEST_ITEMS)

KEYS=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
CYR2LAT=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi")
LAT2CYR=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
URE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
UM={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QRE=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]
WORD=re.compile(r"[0-9a-zа-я]+",re.I)

def attrs(a):
    try:return json.loads(a) if isinstance(a,str) else (a if isinstance(a,dict) else {})
    except:return {}

def norm(x):return re.sub(r"[^0-9a-zа-я]+"," ",str(x).lower().replace("ё","е")).strip()
def compact(x):return re.sub(r"[^0-9a-zа-я]+","",str(x).lower().replace("ё","е"))

def fixmix(t):
    c=sum("\u0400"<=x<="\u04ff" for x in t); l=sum(x.isascii() and x.isalpha() for x in t)
    return t.translate(CYR2LAT if l>=c else LAT2CYR) if c and l else t

def units(s):
    z=set()
    for m in URE.finditer(s):
        u,k=UM[m.group(2).lower()]; z.add(f"{float(m.group(1).replace(',','.'))*k:g}{u}")
    return frozenset(z)

def quantity(s):
    m=QRE[2].search(s)
    if m:return int(m.group(1))*int(m.group(2))
    for r in (QRE[0],QRE[1],QRE[3]):
        m=r.search(s)
        if m:return int(m.group(1))

def build_text(name,a):
    d=attrs(a); parts=[str(name or "")]
    if d:
        low={str(k).lower():str(v) for k,v in d.items() if v}; picked=[]; used=set()
        for w in KEYS:
            for k,v in low.items():
                if w in k and k not in used:picked.append(f"{k}:{v}");used.add(k)
        parts.append(" ; ".join(picked+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    s=" | ".join(parts).replace("ё","е").replace("Ё","Е")
    s=" ".join(fixmix(x) for x in s.split()); s=re.sub(r"[×хХ](?=\d)","x",s)
    e=units(s); q=quantity(s.lower()); extra=[]
    if e:extra.append("ед: "+" ".join(sorted(e)[:12]))
    if q and 1<q<=1000:extra.append(f"кол-во: {q}")
    return (s+(" | "+" | ".join(extra) if extra else ""))[:2000]

ALIASES={
"brand":("бренд","brand","производитель"),"model":("модель","model"),
"article":("артикул","партномер","oem","код производителя"),
"size":("размер","size"),"color":("цвет","color"),
"material":("материал","material"),"gender":("пол ","gender"),
"memory":("память","накопитель"),"pack":("количество","комплектация","упаковка")}

def profile(i,n,a,c):
    d={norm(k):str(v) for k,v in attrs(a).items() if v}; text=build_text(n,a)
    s=norm(text+" "+" ".join(d.values())); w=frozenset(WORD.findall(s))
    code=frozenset(x for x in re.findall(r"[0-9a-zа-я][0-9a-zа-я._/-]{2,}",s) if any(q.isdigit() for q in x) and any(q.isalpha() for q in x))
    num=frozenset(f"{float(x.replace(',','.')):g}" for x in re.findall(r"\d+(?:[.,]\d+)?",s))
    fs={k:frozenset(compact(v) for x,v in d.items() if any(z in x for z in al) and compact(v)) for k,al in ALIASES.items()}
    q=quantity(s)
    if q:fs["pack"]=fs["pack"]|{str(q)}
    cn=compact(n); tri=frozenset(cn[j:j+3] for j in range(max(0,len(cn)-2)))
    return {"text":text,"name":cn,"tok":w,"tri":tri,"num":num,"code":code,"unit":units(s),
            "keys":frozenset(d),"vals":frozenset(WORD.findall(" ".join(d.values()))),
            "fields":fs,"ac":len(d),"cat":str(c or "")}

def load_profiles(path,need):
    t=time.time()
    d=(pl.scan_parquet(path).select(["id","name","attributes","category"])
       .join(need.lazy(),on="id",how="semi").collect(streaming=True))
    p={i:profile(i,n,a,c) for i,n,a,c in d.iter_rows()}
    missing=set(need["id"].to_list())-set(p)
    if missing:raise ValueError(f"Не найдено товаров: {len(missing)}")
    print(f"profiles {len(p):,}: {time.time()-t:.1f}s")
    return p

train=pl.read_parquet(MATCHES,columns=["id1","id2","target"])
need=pl.concat([train.select(pl.col("id1").alias("id")),train.select(pl.col("id2").alias("id"))]).unique()
P=load_profiles(ITEMS,need)

tok=AutoTokenizer.from_pretrained(MODEL,local_files_only=True)
net=AutoModelForSequenceClassification.from_pretrained(MODEL,torch_dtype=torch.float16,local_files_only=True).cuda().eval()
if torch.cuda.device_count()>1:net=torch.nn.DataParallel(net,device_ids=list(range(torch.cuda.device_count())))
torch.backends.cuda.matmul.allow_tf32=True

def bge_score(df,p,cache,bs=512):
    if cache.exists():
        z=np.load(cache)
        if len(z)==len(df):print("cache:",cache);return z
    a=df["id1"].to_list(); b=df["id2"].to_list(); n=len(a)
    order=np.argsort([len(p[x]["text"])+len(p[y]["text"]) for x,y in zip(a,b)],kind="stable")
    pred=np.empty(n,np.float32); pos=0; t=time.time()
    with torch.inference_mode():
        while pos<n:
            ix=order[pos:pos+bs]; x=[p[a[i]]["text"] for i in ix]; y=[p[b[i]]["text"] for i in ix]
            try:
                z=0
                for u,v in ((x,y),(y,x)):
                    e=tok(u,v,padding=True,truncation=True,max_length=320,pad_to_multiple_of=8,return_tensors="pt")
                    e={k:q.cuda(non_blocking=True) for k,q in e.items()}
                    z+=torch.sigmoid(net(**e).logits.squeeze(-1).float()).cpu().numpy()
                pred[ix]=z/2; pos+=len(ix)
                if pos%20000<len(ix):print(f"BGE {pos:,}/{n:,}, {pos/(time.time()-t):.0f} pair/s")
            except RuntimeError as e:
                if "out of memory" not in str(e).lower() or bs<=64:raise
                bs//=2;torch.cuda.empty_cache();print("OOM -> batch",bs)
    np.save(cache,pred);return pred

bge_train=bge_score(train,P,OUT/"bge_human.npy")

SETS=["num","code","unit","keys","vals"]; FIELDS=list(ALIASES)
FN=["name_exact","name_len_ratio","token_jaccard","token_containment","token_conflict","token_equal","trigram_jaccard"]
for k in SETS:FN += [f"{k}_jaccard",f"{k}_containment",f"{k}_conflict",f"{k}_equal"]
for k in FIELDS:FN += [f"{k}_present",f"{k}_equal",f"{k}_conflict",f"{k}_jaccard"]
FN+=["attr_count_diff","attr_count_ratio","conflict_count","bge_score","bge_x_conflict"]

def stat(a,b):
    inter=len(a&b); union=len(a|b)
    return [inter/union if union else 0,inter/max(1,min(len(a),len(b))) if a and b else 0,
            float(bool(a) and bool(b) and not inter),float(bool(a) and a==b)]

def make_X(df,p,bge):
    A=df["id1"].to_list();B=df["id2"].to_list();X=np.empty((len(df),len(FN)),np.float32);cats=[]
    for i,(x,y) in enumerate(zip(A,B)):
        a,b=p[x],p[y]; ts=stat(a["tok"],b["tok"]); vals=[float(a["name"]==b["name"]),min(len(a["name"]),len(b["name"]))/max(1,max(len(a["name"]),len(b["name"]))),*ts,stat(a["tri"],b["tri"])[0]]; conflicts=ts[2]
        for k in SETS:
            z=stat(a[k],b[k]);vals+=z;conflicts+=z[2]
        for k in FIELDS:
            z=stat(a["fields"][k],b["fields"][k]);vals += [float(bool(a["fields"][k]) and bool(b["fields"][k])),z[3],z[2],z[0]];conflicts+=z[2]
        vals += [abs(a["ac"]-b["ac"]),min(a["ac"],b["ac"])/max(1,max(a["ac"],b["ac"])),conflicts,bge[i],bge[i]*min(conflicts,5)]
        X[i]=vals;cats.append(a["cat"])
    z=pd.DataFrame(X,columns=FN);z["category"]=cats
    return z,np.asarray(cats)

X,cats=make_X(train,P,bge_train);del P;gc.collect()

HAS_TEST=TEST_MATCHES is not None and TEST_ITEMS is not None
if HAS_TEST:
    test=pl.read_parquet(TEST_MATCHES,columns=["id1","id2"])
    need=pl.concat([test.select(pl.col("id1").alias("id")),test.select(pl.col("id2").alias("id"))]).unique()
    PT=load_profiles(TEST_ITEMS,need)
    bge_test=bge_score(test,PT,OUT/"bge_test.npy")
    XT,test_cats=make_X(test,PT,bge_test)
    del PT;gc.collect()

del net,tok;gc.collect();torch.cuda.empty_cache()

def group_val(df,frac=.2,seed=42):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    for a,b in zip(df["id1"],df["id2"]):
        a,b=find(a),find(b)
        if a!=b:parent[b]=a
    comp=np.fromiter((find(x) for x in df["id1"]),dtype=np.int64,count=len(df))
    u=np.unique(comp);rng=np.random.RandomState(seed);vs=set(u[rng.rand(len(u))<frac])
    return np.fromiter((x in vs for x in comp),bool,len(comp))

y=train["target"].to_numpy().astype(np.int8);val=group_val(train);tr=~val
ci=FN.index("conflict_count")

def weights(mask):
    cc=pd.Series(cats[mask]).value_counts();w=np.array([1/np.sqrt(cc[c]) for c in cats[mask]])
    w/=w.mean(); yy=y[mask];bb=bge_train[mask];cf=X.iloc[np.where(mask)[0],ci].to_numpy()
    return w*(1+3*((yy==0)&(bb>.35))+2*((yy==0)&(bb>.35)&(cf>0)))

params=dict(loss_function="Logloss",eval_metric="PRAUC",iterations=1800,depth=8,learning_rate=.04,
            l2_leaf_reg=8,random_strength=.5,bootstrap_type="Bernoulli",subsample=.85,
            border_count=128,random_seed=42,task_type="GPU",devices="0:1",allow_writing_files=False,verbose=100)

model=CatBoostClassifier(**params)
model.fit(Pool(X.iloc[tr],y[tr],weight=weights(tr),cat_features=["category"]),
          eval_set=Pool(X.iloc[val],y[val],cat_features=["category"]),early_stopping_rounds=150,use_best_model=True)
expert_val=model.predict_proba(X.iloc[val])[:,1]

def rank(x):return pd.Series(x).rank(method="average",pct=True).to_numpy()
def macro(y,p,c):
    return np.mean([average_precision_score(y[c==k],p[c==k]) for k in np.unique(c) if len(np.unique(y[c==k]))>1])

yv,cv,bv=y[val],cats[val],bge_train[val]
blend=np.empty(len(yv));alphas={};bycat={}
for c in np.unique(cv):
    m=cv==c; rb,rex=rank(bv[m]),rank(expert_val[m]); base=average_precision_score(yv[m],rb)
    scores=[average_precision_score(yv[m],(1-a)*rb+a*rex) for a in np.arange(0,1.01,.1)]
    j=int(np.argmax(scores));a=j/10
    if scores[j]-base<.0015:a=0
    blend[m]=(1-a)*rb+a*rex;alphas[c]=a
    bycat[c]={"pairs":int(m.sum()),"bge":base,"expert":average_precision_score(yv[m],rex),"blend":average_precision_score(yv[m],blend[m]),"alpha":a}

metrics={"bge_macro":macro(yv,bv,cv),"expert_macro":macro(yv,expert_val,cv),"blend_macro":macro(yv,blend,cv),"best_iteration":model.get_best_iteration()+1,"by_category":bycat,"features":FN}
print(json.dumps({k:v for k,v in metrics.items() if k!="by_category" and k!="features"},indent=2))
(OUT/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2))
(OUT/"category_alpha.json").write_text(json.dumps(alphas,ensure_ascii=False,indent=2))

vi=np.where(val)[0]
pl.DataFrame({"id1":train["id1"][vi],"id2":train["id2"][vi],"target":yv,"category":cv,
              "bge_score":bv,"conflict_score":expert_val,"predict":blend}).write_parquet(OUT/"conflict_val.parquet")

best=max(300,model.get_best_iteration()+1);params["iterations"]=best
final=CatBoostClassifier(**params)
final.fit(Pool(X,y,weight=weights(np.ones(len(y),bool)),cat_features=["category"]))
final.save_model(OUT/"conflict_expert.cbm")

if HAS_TEST:
    ep=final.predict_proba(XT)[:,1];pred=np.empty(len(test))
    for c in np.unique(test_cats):
        m=test_cats==c;a=alphas.get(c,0);pred[m]=(1-a)*rank(bge_test[m])+a*rank(ep[m])
    pl.DataFrame({"id1":test["id1"],"id2":test["id2"],"category":test_cats,
                  "bge_score":bge_test,"conflict_score":ep,"predict":pred}).write_parquet(OUT/"conflict_test.parquet")
    pl.DataFrame({"id1":test["id1"],"id2":test["id2"],"predict":pred}).write_csv(OUT/"conflict_submission.csv")
    print("Готов test:",OUT/"conflict_submission.csv")
else:
    print("Test-файлы не подключены: модель и validation сохранены.")

print("Файлы:",[p.name for p in OUT.iterdir()])

GPU: ['Tesla T4', 'Tesla T4']
train: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/matches.parquet /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/items_human.parquet 
model: /kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1 
test: None None


/tmp/ipykernel_58/788374799.py:106: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .join(need.lazy(),on="id",how="semi").collect(streaming=True))


profiles 711,304: 577.8s


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE 20,480/365,654, 198 pair/s
BGE 40,448/365,654, 162 pair/s
BGE 60,416/365,654, 141 pair/s
BGE 80,384/365,654, 127 pair/s
BGE 100,352/365,654, 117 pair/s
BGE 120,320/365,654, 108 pair/s
BGE 140,288/365,654, 100 pair/s
BGE 160,256/365,654, 94 pair/s
BGE 180,224/365,654, 90 pair/s
BGE 200,192/365,654, 87 pair/s
BGE 220,160/365,654, 85 pair/s
BGE 240,128/365,654, 83 pair/s
BGE 260,096/365,654, 81 pair/s
BGE 280,064/365,654, 79 pair/s
BGE 300,032/365,654, 78 pair/s
BGE 320,000/365,654, 77 pair/s
BGE 340,480/365,654, 76 pair/s
BGE 360,448/365,654, 75 pair/s


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4933269	test: 0.7226112	best: 0.7226112 (0)	total: 177ms	remaining: 5m 18s
100:	learn: 0.5706566	test: 0.7579459	best: 0.7587688 (38)	total: 5.35s	remaining: 1m 29s
200:	learn: 0.5977710	test: 0.7587185	best: 0.7591874 (175)	total: 10.4s	remaining: 1m 22s
300:	learn: 0.6180411	test: 0.7595805	best: 0.7597236 (276)	total: 15.4s	remaining: 1m 16s
400:	learn: 0.6355531	test: 0.7603283	best: 0.7603649 (387)	total: 20.6s	remaining: 1m 11s
500:	learn: 0.6511520	test: 0.7610243	best: 0.7610243 (500)	total: 25.6s	remaining: 1m 6s
600:	learn: 0.6650355	test: 0.7622881	best: 0.7622881 (600)	total: 30.6s	remaining: 1m 1s
700:	learn: 0.6768491	test: 0.7628743	best: 0.7628956 (698)	total: 35.6s	remaining: 55.8s
800:	learn: 0.6879452	test: 0.7637932	best: 0.7638117 (797)	total: 40.7s	remaining: 50.8s
900:	learn: 0.6985007	test: 0.7644458	best: 0.7644458 (900)	total: 45.7s	remaining: 45.6s
1000:	learn: 0.7082141	test: 0.7649732	best: 0.7649732 (1000)	total: 50.8s	remaining: 40.5s
1100:	le

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4924098	total: 98.9ms	remaining: 2m 57s
100:	learn: 0.5697982	total: 4.81s	remaining: 1m 20s
200:	learn: 0.5941496	total: 9.27s	remaining: 1m 13s
300:	learn: 0.6122407	total: 13.8s	remaining: 1m 8s
400:	learn: 0.6284226	total: 18.3s	remaining: 1m 3s
500:	learn: 0.6424865	total: 22.8s	remaining: 59.1s
600:	learn: 0.6551206	total: 27.4s	remaining: 54.6s
700:	learn: 0.6661771	total: 31.9s	remaining: 50s
800:	learn: 0.6760263	total: 36.5s	remaining: 45.5s
900:	learn: 0.6853259	total: 41s	remaining: 40.9s
1000:	learn: 0.6939197	total: 45.6s	remaining: 36.4s
1100:	learn: 0.7017989	total: 50.1s	remaining: 31.8s
1200:	learn: 0.7093371	total: 54.6s	remaining: 27.3s
1300:	learn: 0.7168644	total: 59.2s	remaining: 22.7s
1400:	learn: 0.7234039	total: 1m 3s	remaining: 18.2s
1500:	learn: 0.7300106	total: 1m 8s	remaining: 13.6s
1600:	learn: 0.7360238	total: 1m 12s	remaining: 9.04s
1700:	learn: 0.7419426	total: 1m 17s	remaining: 4.5s
1799:	learn: 0.7478285	total: 1m 21s	remaining: 0us
Test-

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import shutil

src=Path("/kaggle/working/conflict_expert")
assert (src/"conflict_expert.cbm").exists(), list(src.glob("*"))

zip_path=shutil.make_archive(
    "/kaggle/working/conflict_expert_backup",
    "zip",
    root_dir=src
)
print(zip_path, f"{Path(zip_path).stat().st_size/1024/1024:.1f} MB")
display(FileLink(zip_path))